In [ ]:
run_id = ""  
pipeline_name = ""

In [2]:
# Input tables
SILVER_PRODUCTS = "silver_products"
SILVER_REVIEWS = "silver_reviews"
SILVER_PRICE_HISTORY = "silver_price_history"

# Output tables (star schema)
DIM_PRODUCTS = "dim_products"
DIM_CATEGORIES = "dim_categories"
DIM_DATE = "dim_date"
FACT_REVIEWS = "fact_reviews"
FACT_PRICE_CHANGES = "fact_price_changes"

# Initialize metastore User Data Functions
metastore = notebookutils.udf.getFunctions('metastore_functions')


StatementMeta(, c319ec64-48ab-40ab-9647-f7a613a128a3, 4, Finished, Available, Finished)

In [3]:
# Cell 2: Build dimension tables from Silver layer
from pyspark.sql import functions as F

silver_products = spark.read.table(SILVER_PRODUCTS)
silver_reviews = spark.read.table(SILVER_REVIEWS)
silver_price_history = spark.read.table(SILVER_PRICE_HISTORY)

# dim_products (directly from silver_products)
dim_products = silver_products.select(
    F.col("product_id").alias("product_key"),
    "product_name", "description", "category_name", "country_of_origin",
    "current_price", "inventory", "inventory_status",
    "first_available", "days_since_available",
    "price_change_pct", "price_change_direction"
)
dim_products.write.mode("overwrite").format("delta").saveAsTable(DIM_PRODUCTS)

# dim_categories (distinct categories with parent/sub split)
dim_categories = (
    silver_products.select("category_name").distinct()
    .withColumn("category_key", F.md5("category_name"))
    .withColumn("parent_category", F.split("category_name", ", ").getItem(0))
    .withColumn("sub_category", F.split("category_name", ", ").getItem(1))
)
dim_categories.write.mode("overwrite").format("delta").saveAsTable(DIM_CATEGORIES)

# dim_date (union of all dates from reviews and price history)
review_dates = silver_reviews.select(F.to_date("review_date").alias("full_date"))
price_dates = silver_price_history.select(F.to_date("price_date").alias("full_date"))
all_dates = review_dates.union(price_dates).distinct()

dim_date = (
    all_dates
    .withColumn("date_key", F.date_format("full_date", "yyyy-MM-dd"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
)
dim_date.write.mode("overwrite").format("delta").saveAsTable(DIM_DATE)

print(f"dim_products: {dim_products.count()}")
print(f"dim_categories: {dim_categories.count()}")
print(f"dim_date: {dim_date.count()}")


StatementMeta(, c319ec64-48ab-40ab-9647-f7a613a128a3, 5, Finished, Available, Finished)

dim_products: 180
dim_categories: 18
dim_date: 456


In [4]:
# Cell 3: Build fact_reviews from silver_reviews joined with dimensions
dim_categories = spark.read.table(DIM_CATEGORIES)

fact_reviews = (
    silver_reviews
    .join(dim_categories.select("category_name", "category_key"), "category_name", "left")
    .select(
        F.col("review_id"),
        F.col("product_id").alias("product_key"),
        F.col("category_key"),
        F.date_format("review_date", "yyyy-MM-dd").alias("date_key"),
        F.col("customer_name"),
        F.col("stars"),
        F.col("sentiment_bucket"),
        F.col("review_text"),
    )
)
fact_reviews.write.mode("overwrite").format("delta").saveAsTable(FACT_REVIEWS)
print(f"fact_reviews: {fact_reviews.count()}")

StatementMeta(, c319ec64-48ab-40ab-9647-f7a613a128a3, 6, Finished, Available, Finished)

fact_reviews: 652


In [5]:
# Cell 4: Build fact_price_changes from silver_price_history joined with dimensions
fact_price_changes = (
    silver_price_history
    .join(dim_categories.select("category_name", "category_key"), "category_name", "left")
    .select(
        F.col("product_id").alias("product_key"),
        F.date_format("price_date", "yyyy-MM-dd").alias("date_key"),
        F.col("category_key"),
        F.col("price"),
        F.col("price_seq"),
        F.col("price_change_pct"),
    )
)
fact_price_changes.write.mode("overwrite").format("delta").saveAsTable(FACT_PRICE_CHANGES)
print(f"fact_price_changes: {fact_price_changes.count()}")

StatementMeta(, c319ec64-48ab-40ab-9647-f7a613a128a3, 7, Finished, Available, Finished)

fact_price_changes: 551


In [6]:
# Cell 5: Print summary and log metadata
import json

try:
    dim_products_count = spark.read.table(DIM_PRODUCTS).count()
    dim_categories_count = spark.read.table(DIM_CATEGORIES).count()
    dim_date_count = spark.read.table(DIM_DATE).count()
    fact_reviews_count = spark.read.table(FACT_REVIEWS).count()
    fact_price_changes_count = spark.read.table(FACT_PRICE_CHANGES).count()

    summary = {
        "run_id": run_id,
        "dim_products_count": dim_products_count,
        "dim_categories_count": dim_categories_count,
        "dim_date_count": dim_date_count,
        "fact_reviews_count": fact_reviews_count,
        "fact_price_changes_count": fact_price_changes_count,
        "status": "success",
    }
    print(json.dumps(summary, indent=2))

    # Log dataset profiles for gold tables
    metastore.log_dataset_profile(
        datasetId="dim_products", runId=run_id, rowCount=dim_products_count,
        columnProfiles=[{"column": "product_key", "nullRate": 0.0}],
        notes="")
    metastore.log_dataset_profile(
        datasetId="fact_reviews", runId=run_id, rowCount=fact_reviews_count,
        columnProfiles=[{"column": "stars", "nullRate": 0.0}],
        notes="")

    # Log transform lineage
    metastore.log_transform_lineage(
        datasetId="dim_products", runId=run_id,
        sourceDatasets=["silver_products"],
        transforms=["select_dimension_columns"],
        columnsAdded=["product_key"])
    metastore.log_transform_lineage(
        datasetId="fact_reviews", runId=run_id,
        sourceDatasets=["silver_reviews", "dim_products"],
        transforms=["join_product_key", "add_sentiment_bucket"],
        columnsAdded=["product_key", "category_key", "date_key"])

except Exception as e:
    raise

StatementMeta(, c319ec64-48ab-40ab-9647-f7a613a128a3, 8, Finished, Available, Finished)

{
  "run_id": "some_id",
  "dim_products_count": 180,
  "dim_categories_count": 18,
  "dim_date_count": 456,
  "fact_reviews_count": 652,
  "fact_price_changes_count": 551,
  "status": "success"
}
